In [1]:
import sys

sys.path.append('/home/guests/jmc/Thesis/original_dreamy/dreamy')
print(sys.path)

from dreamy.epo import epo, add_fwd_hooks, build_pareto_frontier
from dreamy.attribution import resample_viz

['/home/guests/jmc/.conda/envs/thenv/lib/python311.zip', '/home/guests/jmc/.conda/envs/thenv/lib/python3.11', '/home/guests/jmc/.conda/envs/thenv/lib/python3.11/lib-dynload', '', '/home/guests/jmc/.conda/envs/thenv/lib/python3.11/site-packages', '/home/guests/jmc/Thesis/original_dreamy/dreamy']


/home/guests/jmc/.conda/envs/thenv/lib/python3.11/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import transformers
import torch
from IPython.display import HTML, display




In [3]:

%config InlineBackend.figure_format='retina'
np.set_printoptions(edgeitems=10, linewidth=100)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)

#| output: false
model_name = "microsoft/phi-2"
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    torch_dtype="auto",
    use_cache=False,
    device_map="cuda"
)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

RuntimeError: The NVIDIA driver on your system is too old (found version 11060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver.

In [ ]:
def neuron_runner(layer, neuron):
    def f(*model_args, **model_kwargs):
        out = {}

        def get_target(module, input, output):
            out["target"] = input[0][:, -1, neuron]

        with add_fwd_hooks(
            [
                (model.model.layers[layer].mlp.fc2, get_target),
            ]
        ):
            out["logits"] = model(*model_args, **model_kwargs).logits
        return out

    return f

In [ ]:
#| output: false
runner = neuron_runner(layer=8, neuron=1)
history = epo(runner, model, tokenizer, seed=1001, restart_xentropy = 0.5, restart_xentropy_max_mult = 3.0)